# Hash Table & Graph

## Hash Table Implementation
Custom hash table using separate chaining for collision handling, with dynamic resizing.

In [6]:
class HashTable:
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.size = 0
        self.buckets = [[] for _ in range(self.capacity)]
        self.LOAD_FACTOR_THRESHOLD = 0.7

    def _hash(self, key):
        return hash(key) % self.capacity

    def _resize(self):
        old_buckets = self.buckets
        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:
            for key, value in bucket:
                self.put(key, value)

    def put(self, key, value):
        if self.size / self.capacity >= self.LOAD_FACTOR_THRESHOLD:
            self._resize()
        index = self._hash(key)
        bucket = self.buckets[index]
        for i, (existing_key, _) in enumerate(bucket):
            if existing_key == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self.size += 1

    def get(self, key):
        index = self._hash(key)
        bucket = self.buckets[index]
        for existing_key, value in bucket:
            if existing_key == key:
                return value
        raise KeyError(key)

    def remove(self, key):
        index = self._hash(key)
        bucket = self.buckets[index]
        for i, (existing_key, _) in enumerate(bucket):
            if existing_key == key:
                del bucket[i]
                self.size -= 1
                return
        raise KeyError(key)

    def contains(self, key):
        try:
            self.get(key)
            return True
        except KeyError:
            return False

    def __len__(self):
        return self.size

    def __repr__(self):
        items = []
        for bucket in self.buckets:
            for key, value in bucket:
                items.append(f"{key!r}: {value!r}")
        return "{" + ", ".join(items) + "}"

## Hash Table — Interactive GUI
Enter a key/value and click a button to Put, Get, Remove, or check Contains. The bucket view below updates live.

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output

ht = HashTable(capacity=4)

key_box = widgets.Text(description="Key:")
value_box = widgets.Text(description="Value:")

put_btn = widgets.Button(description="Put", button_style="success")
get_btn = widgets.Button(description="Get", button_style="info")
remove_btn = widgets.Button(description="Remove", button_style="danger")
contains_btn = widgets.Button(description="Contains?", button_style="warning")

output = widgets.Output()

def refresh_view(message=""):
    with output:
        clear_output()
        if message:
            print(message)
        print(f"\nSize: {len(ht)} | Capacity: {ht.capacity}")
        for i, bucket in enumerate(ht.buckets):
            print(f"Bucket {i}: {bucket}")

def on_put(b):
    ht.put(key_box.value, value_box.value)
    refresh_view(f"Put ({key_box.value!r} -> {value_box.value!r})")

def on_get(b):
    try:
        result = ht.get(key_box.value)
        refresh_view(f"get({key_box.value!r}) -> {result!r}")
    except KeyError:
        refresh_view(f"get({key_box.value!r}) -> KeyError (not found)")

def on_remove(b):
    try:
        ht.remove(key_box.value)
        refresh_view(f"Removed {key_box.value!r}")
    except KeyError:
        refresh_view(f"Cannot remove {key_box.value!r} — not found")

def on_contains(b):
    result = ht.contains(key_box.value)
    refresh_view(f"contains({key_box.value!r}) -> {result}")

put_btn.on_click(on_put)
get_btn.on_click(on_get)
remove_btn.on_click(on_remove)
contains_btn.on_click(on_contains)

display(widgets.HBox([key_box, value_box]))
display(widgets.HBox([put_btn, get_btn, remove_btn, contains_btn]))
display(output)
refresh_view("Ready.")

Output()

## Graph Implementation
Adjacency list-based graph supporting directed/undirected and weighted edges, with BFS, DFS, and shortest path.

In [8]:
class Graph:
    def __init__(self, directed=False):
        self.directed = directed
        self.adjacency = {}

    def add_node(self, node):
        if node not in self.adjacency:
            self.adjacency[node] = []

    def add_edge(self, u, v, weight=1):
        self.add_node(u)
        self.add_node(v)
        self.adjacency[u].append((v, weight))
        if not self.directed:
            self.adjacency[v].append((u, weight))

    def neighbors(self, node):
        return self.adjacency.get(node, [])

    def bfs(self, start):
        visited = {start}
        queue = [start]
        order = []
        head = 0
        while head < len(queue):
            node = queue[head]
            head += 1
            order.append(node)
            for neighbor, _ in self.neighbors(node):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        return order

    def dfs(self, start):
        visited = set()
        stack = [start]
        order = []
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            order.append(node)
            for neighbor, _ in reversed(self.neighbors(node)):
                if neighbor not in visited:
                    stack.append(neighbor)
        return order

    def shortest_path_unweighted(self, start, end):
        if start == end:
            return [start]
        visited = {start}
        queue = [start]
        parent = {start: None}
        head = 0
        while head < len(queue):
            node = queue[head]
            head += 1
            for neighbor, _ in self.neighbors(node):
                if neighbor not in visited:
                    visited.add(neighbor)
                    parent[neighbor] = node
                    if neighbor == end:
                        path = [end]
                        while parent[path[-1]] is not None:
                            path.append(parent[path[-1]])
                        return path[::-1]
                    queue.append(neighbor)
        return None

    def __repr__(self):
        lines = []
        for node, edges in self.adjacency.items():
            edge_str = ", ".join(f"{n}(w={w})" for n, w in edges)
            lines.append(f"{node} -> [{edge_str}]")
        return "\n".join(lines)

## Graph — Interactive GUI
Add edges, then run BFS/DFS/Shortest Path from a chosen start node. The graph is redrawn after every change.

In [9]:
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

g = Graph(directed=False)

u_box = widgets.Text(description="Node U:")
v_box = widgets.Text(description="Node V:")
w_box = widgets.Text(description="Weight:", value="1")
add_edge_btn = widgets.Button(description="Add Edge", button_style="success")

start_box = widgets.Text(description="Start:")
end_box = widgets.Text(description="End:")
bfs_btn = widgets.Button(description="BFS", button_style="info")
dfs_btn = widgets.Button(description="DFS", button_style="info")
path_btn = widgets.Button(description="Shortest Path", button_style="warning")

graph_output = widgets.Output()
text_output = widgets.Output()

def draw_graph(highlight_path=None):
    with graph_output:
        clear_output(wait=True)
        G = nx.Graph() if not g.directed else nx.DiGraph()
        for node, edges in g.adjacency.items():
            G.add_node(node)
            for neighbor, weight in edges:
                G.add_edge(node, neighbor, weight=weight)

        if len(G.nodes) == 0:
            print("Graph is empty — add an edge to begin.")
            return

        pos = nx.spring_layout(G, seed=42)
        plt.figure(figsize=(5, 4))

        edge_colors = []
        path_edges = set()
        if highlight_path and len(highlight_path) > 1:
            path_edges = set(zip(highlight_path, highlight_path[1:]))
            path_edges |= set(zip(highlight_path[1:], highlight_path))

        for edge in G.edges():
            edge_colors.append("red" if edge in path_edges else "gray")

        nx.draw(G, pos, with_labels=True, node_color="lightblue",
                node_size=800, edge_color=edge_colors, width=2)
        edge_labels = nx.get_edge_attributes(G, "weight")
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
        plt.show()

def on_add_edge(b):
    try:
        weight = int(w_box.value) if w_box.value else 1
    except ValueError:
        weight = 1
    g.add_edge(u_box.value, v_box.value, weight)
    draw_graph()
    with text_output:
        clear_output()
        print(f"Added edge {u_box.value} - {v_box.value} (weight={weight})")

def on_bfs(b):
    order = g.bfs(start_box.value)
    draw_graph()
    with text_output:
        clear_output()
        print("BFS order:", order)

def on_dfs(b):
    order = g.dfs(start_box.value)
    draw_graph()
    with text_output:
        clear_output()
        print("DFS order:", order)

def on_path(b):
    path = g.shortest_path_unweighted(start_box.value, end_box.value)
    draw_graph(highlight_path=path)
    with text_output:
        clear_output()
        print("Shortest path:", path if path else "No path found")

add_edge_btn.on_click(on_add_edge)
bfs_btn.on_click(on_bfs)
dfs_btn.on_click(on_dfs)
path_btn.on_click(on_path)

display(widgets.HBox([u_box, v_box, w_box, add_edge_btn]))
display(widgets.HBox([start_box, end_box, bfs_btn, dfs_btn, path_btn]))
display(text_output)
display(graph_output)
draw_graph()

Output()

Output()